# Random Forest


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

from kaggle_games.pipelines import get_rf_pipeline
from kaggle_games.split_data import get_split_data

In [ ]:
data_path = Path.cwd().parent / "data" / "games.json"

df = pd.read_json(data_path, orient="index")
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

X_train, X_test, y_train, y_test = get_split_data(df)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

prep_steps = get_rf_pipeline().steps

pipe_baseline = Pipeline(
    steps=prep_steps
    + [("rf", RandomForestClassifier(random_state=42, n_jobs=-1))]
)

pipe_cw = Pipeline(
    steps=prep_steps
    + [
        (
            "rf",
            RandomForestClassifier(
                class_weight="balanced", random_state=42, n_jobs=-1
            ),
        )
    ]
)

pipe_smote = ImbPipeline(
    steps=prep_steps
    + [
        ("smote", SMOTE(random_state=42)),
        ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ]
)

param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [10, 20],
}


def run_search(pipeline, name):
    print(f"--- Tuning {name} ---")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter=3,
        cv=cv_strategy,
        scoring="roc_auc",
        random_state=42,
        n_jobs=None,
        verbose=1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_


best_baseline = run_search(pipe_baseline, "Baseline")
best_cw = run_search(pipe_cw, "Class Weights")
best_smote = run_search(pipe_smote, "SMOTE")

models = {
    "Baseline": best_baseline,
    "Class Weights": best_cw,
    "SMOTE": best_smote,
}


def evaluate_model(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {title} ===")
    print(classification_report(y_test, y_pred))

    return {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "auc": roc_auc_score(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
        "prec": precision_score(y_test, y_pred),
        "rec": recall_score(y_test, y_pred),
    }


eval_results = {}
for name, model in models.items():
    eval_results[name] = evaluate_model(model, X_test, y_test, name)

plt.figure(figsize=(10, 7))
colors = {"Baseline": "gray", "Class Weights": "blue", "SMOTE": "red"}

for name, res in eval_results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {res['auc']:.3f})",
        color=colors[name],
        linewidth=2,
    )

plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.title("ROC Curve Comparison", fontweight="bold", fontsize=14)
plt.legend()
plt.show()

plt.figure(figsize=(10, 7))
for name, res in eval_results.items():
    precision, recall, _ = precision_recall_curve(y_test, res["y_proba"])
    plt.plot(
        recall,
        precision,
        label=f"{name} (F1 = {res['f1']:.3f})",
        color=colors[name],
        linewidth=2,
    )

plt.title("Precision-Recall Curve Comparison", fontweight="bold", fontsize=14)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, res) in enumerate(eval_results.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, res["y_pred"], ax=axes[i], cmap="Purples", colorbar=False
    )
    axes[i].set_title(f"{name}\nF1: {res['f1']:.2f} | Recall: {res['rec']:.2f}")

plt.tight_layout()
plt.show()